In [7]:
import sqlite3
# Input main database file
database_file = "C:/Users/twans/Documents/GitHub/NBA/Tao Data Collection/1-13_nba_data.db.db"
# Connect to the primary database
conn = sqlite3.connect(database_file)
cursor = conn.cursor()

# Attach nba_odds_event_ids
cursor.execute("ATTACH DATABASE 'nba_odds_event_ids.db' AS secondary")

# Attach tables from secondary db
cursor.execute("SELECT name FROM secondary.sqlite_master WHERE type='table'")
print(cursor.fetchall())

# Copy the entire table from secondary to a new table in main database
create_copy_query = """
CREATE TABLE nba_odds_event_ids AS
SELECT *
FROM secondary.events
"""
cursor.execute(create_copy_query)
conn.commit()
cursor.execute("DETACH DATABASE secondary")

[('events',)]


In [9]:
# Connect to the primary database
conn = sqlite3.connect(database_file)
cursor = conn.cursor()

# Attach the secondary database
cursor.execute("ATTACH 'historical_player_odds.db' AS secondary")

# List all tables in the secondary database
cursor.execute("SELECT name FROM secondary.sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print("Tables in secondary DB:", tables)

# List of tables to copy
tables_to_copy = ['draftkings_player_points', 'fan_duel_player_points']  # Replace with your selected table names

# Iterate through the selected tables and copy them
for table in tables_to_copy:
    copy_query = f"""
    CREATE TABLE {table} AS
    SELECT *
    FROM secondary.{table};
    """
    cursor.execute(copy_query)

# Commit and detach
conn.commit()
cursor.execute("DETACH DATABASE secondary")

Tables in secondary DB: [('odds',), ('betmgm_Odds',), ('betonlineag_Odds',), ('betrivers_Odds',), ('betus_Odds',), ('bovada_Odds',), ('draftkings_Odds',), ('fanduel_Odds',), ('mybookieag_Odds',), ('pointsbetus_Odds',), ('unibet_us_Odds',), ('williamhill_us_Odds',), ('wynnbet_Odds',), ('barstool_Odds',), ('refined_odds',), ('fan_duel_player_points',), ('draftkings_player_points',)]


In [14]:

# Connect to your SQLite database
conn = sqlite3.connect(database_file)
cursor = conn.cursor()

# Add a new column named REGULAR_DATE of type DATE
try:
    cursor.execute("""
    ALTER TABLE nba_odds_event_ids
    ADD COLUMN REGULAR_DATE DATE
    """)
except sqlite3.OperationalError as e:
    print("Warning:", e)

# Populate the REGULAR_DATE column using the commence_time column,
# converting the ISO timestamp to CST by subtracting 6 hours.
cursor.execute("""
UPDATE nba_odds_event_ids
SET REGULAR_DATE = date(datetime(commence_time, '-6 hours'))
""")

# Commit changes after updating the new column
conn.commit()

# Step 3: Delete rows where REGULAR_DATE is before May 4th, 2023
cursor.execute("""
DELETE FROM nba_odds_event_ids
WHERE REGULAR_DATE < '2023-05-04'
""")

# Commit the deletion
conn.commit()
print('created standardized date')
# Close the connection
cursor.close()
conn.close()


In [18]:
# Connect to your SQLite database
conn = sqlite3.connect(database_file)
cursor = conn.cursor()

# Step 1: Add a new column named REGULAR_DATE of type DATE
try:
    cursor.execute("""
    ALTER TABLE combined_player_game_data
    ADD COLUMN REGULAR_DATE DATE
    """)
except sqlite3.OperationalError as e:
    print("Warning:", e)

# Step 2: Populate the REGULAR_DATE column using the GAME_DATE_EST column
cursor.execute("""
UPDATE combined_player_game_data
SET REGULAR_DATE = date(substr(GAME_DATE_EST, 1, 10))
""")

# Commit changes after updating the new column
conn.commit()
print('created standardized date')
# Close the connection
cursor.close()
conn.close()


created standardized date


In [20]:
# Connect to your SQLite database
conn = sqlite3.connect(database_file)
cursor = conn.cursor()

cursor.execute("""
    ALTER TABLE combined_player_game_data
    ADD COLUMN HOME_TEAM
    """)

cursor.execute("""
    ALTER TABLE combined_player_game_data
    ADD COLUMN AWAY_TEAM
    """)

# Step 2: Update the new columns based on GAMECODE substring
try:
    cursor.execute("""
    UPDATE combined_player_game_data
    SET 
        HOME_TEAM = substr(GAMECODE, length(GAMECODE) - 5, 3),
        AWAY_TEAM = substr(GAMECODE, length(GAMECODE) - 2, 3)
    WHERE GAMECODE IS NOT NULL
    """)
    conn.commit()
except sqlite3.OperationalError as e:
    print("Error updating table with team columns:", e)

# Commit changes after updating the new column
conn.commit()
print('created home, away team columns')
# Close the connection
cursor.close()
conn.close()

In [22]:
# Define mapping for team names to abbreviations
team_abbreviations = {
    "Atlanta Hawks": "ATL",
    "Boston Celtics": "BOS",
    "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA",
    "Chicago Bulls": "CHI",
    "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL",
    "Denver Nuggets": "DEN",
    "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW",
    "Houston Rockets": "HOU",
    "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC",
    "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA",
    "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP",
    "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL",
    "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR",
    "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR",
    "Utah Jazz": "UTA",
    "Washington Wizards": "WAS"
}


# Connect to database
conn = sqlite3.connect(database_file)
cursor = conn.cursor()

table_name = "nba_odds_event_ids"

# Function to add a column if it doesn't exist
def add_column_if_not_exists(cursor, table, column, col_type="TEXT"):
    try:
        cursor.execute(f"ALTER TABLE {table} ADD COLUMN {column} {col_type}")
        print(f"Added column {column}")
    except sqlite3.OperationalError as e:
        if "duplicate column name" in str(e).lower():
            print(f"Column {column} already exists, skipping.")
        else:
            print(f"Error adding column {column}: {e}")

# Step 1: Add new columns for abbreviations
add_column_if_not_exists(cursor, table_name, "HOME_TEAM_ABBR")
add_column_if_not_exists(cursor, table_name, "AWAY_TEAM_ABBR")

# Step 2: Retrieve all distinct home and away team names from the table
cursor.execute(f"SELECT DISTINCT home_team, away_team FROM {table_name}")
teams = cursor.fetchall()

# Step 3: Update abbreviation columns using the mapping
for home_team, away_team in teams:
    home_code = team_abbreviations.get(home_team, None)
    away_code = team_abbreviations.get(away_team, None)
    
    # Update rows matching the current pair of home and away teams
    update_query = f"""
    UPDATE {table_name}
    SET 
        HOME_TEAM_ABBR = ?,
        AWAY_TEAM_ABBR = ?
    WHERE home_team = ? AND away_team = ?
    """
    cursor.execute(update_query, (home_code, away_code, home_team, away_team))

conn.commit()
print("Updated home_abbr and away_abbr columns based on team names.")

# Close connection
cursor.close()
conn.close()

Added column HOME_TEAM_ABBR
Added column AWAY_TEAM_ABBR
Updated home_abbr and away_abbr columns based on team names.


In [24]:
# Connect database
conn = sqlite3.connect(database_file)
cursor = conn.cursor()

# Create Combined Table with event ids
cursor.execute("""CREATE TABLE prop_events_and_players AS
SELECT DISTINCT 
    cpg.*, 
    ei.*
FROM 
    combined_player_game_data AS cpg
INNER JOIN
    nba_odds_event_ids AS ei
ON 
    cpg.REGULAR_DATE = ei.REGULAR_DATE 
    AND cpg.HOME_TEAM = ei.AWAY_TEAM_ABBR
    AND cpg.AWAY_TEAM = ei.HOME_TEAM_ABBR;
""")
conn.commit()
print("joined team_data")

# Close connection
cursor.close()
conn.close()

joined team_data


In [28]:
# Connect to your SQLite database
conn = sqlite3.connect(database_file)
cursor = conn.cursor()

# Final combine with fanduel props
cursor.execute("""CREATE TABLE combined_fanduel_prop_and_players AS
SELECT DISTINCT 
    pep.*, 
    fd.*
FROM 
    prop_events_and_players AS pep
INNER JOIN
    fan_duel_player_points AS fd
ON 
    pep.id = fd.event_id
    AND pep.PLAYER_NAME = fd.description
""")
conn.commit()
print("joined data")

# Close connection
cursor.close()
conn.close()

joined data
